# CPAWS DLinear — repository launcher
Source và clean data là hai Kaggle Inputs riêng. Dùng source repository sau khi PR #14 và DLinear đã được tích hợp; không cần package Kaggle cũ và không đưa Parquet vào Git. Notebook chỉ gọi modules/CLI của repository. Chưa có validation 2019.

Xem `docs/models/dlinear.md`. CPU là mặc định; GPU cần được kiểm chứng trên môi trường của bạn. Chạy từ session sạch để tránh imports của source cũ.

## 1. Tìm repository trong Input và tạo bản chạy trong Working

In [ ]:
import os
import shutil
import sys
import uuid
import zipfile
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("CPAWS_INPUT_ROOT", "/kaggle/input"))
WORKING_ROOT = Path(os.environ.get("CPAWS_WORKING_ROOT", "/kaggle/working"))
# Nếu có nhiều bản, thay None bằng đúng đường dẫn được in ra, không đoán tên dataset.
REPOSITORY_OVERRIDE = None  # thư mục source đã giải nén hoặc ZIP repository hoàn chỉnh
WORKING_ROOT.mkdir(parents=True, exist_ok=True)
if "src.forecast.dlinear" in sys.modules:
    raise RuntimeError("Restart session trước khi đổi/reload source repository.")
source_dirs = sorted({p.parents[2] for p in INPUT_ROOT.rglob("src/forecast/train_dlinear.py")})
source_zips = []
for path in INPUT_ROOT.rglob("*.zip"):
    with zipfile.ZipFile(path) as archive:
        if any(name.endswith("src/forecast/train_dlinear.py") for name in archive.namelist()):
            source_zips.append(path)
print("Repository candidates:", [str(p) for p in source_dirs + source_zips])
choices = [Path(REPOSITORY_OVERRIDE)] if REPOSITORY_OVERRIDE else source_dirs + source_zips
if len(choices) != 1:
    raise RuntimeError(
        "Add Input source repository hoàn chỉnh; chọn REPOSITORY_OVERRIDE nếu có nhiều bản."
    )
source = choices[0]
SOURCE_DEST = WORKING_ROOT / ("cpaws-source-" + uuid.uuid4().hex[:8])
if source.is_dir():
    shutil.copytree(
        source,
        SOURCE_DEST,
        ignore=shutil.ignore_patterns(
            ".git",
            ".venv",
            "__pycache__",
            ".pytest_cache",
            ".ruff_cache",
            "*.parquet",
            "*.npz",
            "*.pt",
            "*.pth",
        ),
    )
    REPO_ROOT = SOURCE_DEST
else:
    SOURCE_DEST.mkdir()
    with zipfile.ZipFile(source) as archive:
        for name in archive.namelist():
            if not (SOURCE_DEST / name).resolve().is_relative_to(SOURCE_DEST.resolve()):
                raise ValueError("Unsafe repository ZIP member")
        archive.extractall(SOURCE_DEST)
    roots = {p.parents[2] for p in SOURCE_DEST.rglob("src/forecast/train_dlinear.py")}
    if len(roots) != 1:
        raise RuntimeError("Không xác định được repository root.")
    REPO_ROOT = roots.pop()
for required in ("src/features/build.py", "src/forecast/returns.py", "src/forecast/dlinear.py"):
    if not (REPO_ROOT / required).is_file():
        raise RuntimeError(
            "Source thiếu dependency: "
            + required
            + "; cần repository đã tích hợp PR #14 và DLinear."
        )
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
os.environ["PYTHONPATH"] = str(REPO_ROOT)
assert sys.version_info >= (3, 11), "Python >=3.11 required"
print("Source:", REPO_ROOT, "Python:", sys.version)

## 2. Dependencies tối thiểu cho training/tests
Không cài toàn bộ ingestion/AWS để train. Internet chỉ dùng khi cần cài dependency; không tải market data. Restart session nếu pip thay package đã import.

In [ ]:
import subprocess
from importlib import metadata

try:
    from packaging.requirements import Requirement
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "packaging>=23"], check=True)
    from packaging.requirements import Requirement


def requirements(path):
    for raw in path.read_text().splitlines():
        line = raw.split("#", 1)[0].strip()
        if line.startswith("-r "):
            yield from requirements(path.parent / line[3:].strip())
        elif line:
            yield line


missing = []
for line in requirements(REPO_ROOT / "requirements-dlinear-test.txt"):
    req = Requirement(line)
    try:
        if metadata.version(req.name) not in req.specifier:
            missing.append(line)
    except metadata.PackageNotFoundError:
        missing.append(line)
if missing:
    print("Install dependencies:", missing)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements-dlinear-test.txt"], check=True
    )
else:
    print("Dependencies already compatible.")

## 3. Tìm clean data riêng trong Input
Chọn cặp prices/rejected cùng snapshot. Nếu có nhiều cặp, điền hai override theo đường dẫn thực tế được in ra. Data ZIP được đọc từ Input và chỉ giải nén hai file cần thiết vào Working.

In [ ]:
PRICES_OVERRIDE = None
REJECTED_OVERRIDE = None
pairs = [
    (p, p.parent / "rejected.parquet")
    for p in INPUT_ROOT.rglob("prices.parquet")
    if (p.parent / "rejected.parquet").is_file()
]
print("Clean data pairs:", [(str(p), str(r)) for p, r in pairs])
if PRICES_OVERRIDE and REJECTED_OVERRIDE:
    PRICES_PATH, REJECTED_PATH = Path(PRICES_OVERRIDE), Path(REJECTED_OVERRIDE)
elif len(pairs) == 1:
    PRICES_PATH, REJECTED_PATH = pairs[0]
elif not pairs:
    zip_pairs = []
    for archive_path in INPUT_ROOT.rglob("*.zip"):
        with zipfile.ZipFile(archive_path) as archive:
            names = set(archive.namelist())
            for name in names:
                sibling = str(Path(name).with_name("rejected.parquet")).replace("\\", "/")
                if name.endswith("prices.parquet") and sibling in names:
                    zip_pairs.append((archive_path, name, sibling))
    print("Data ZIP candidates:", [(str(p), a, b) for p, a, b in zip_pairs])
    if len(zip_pairs) != 1:
        raise RuntimeError("Add Input clean snapshot hoặc giải nén rồi cấu hình hai override.")
    archive_path, price_member, reject_member = zip_pairs[0]
    DATA_DEST = WORKING_ROOT / ("cpaws-clean-" + uuid.uuid4().hex[:8])
    DATA_DEST.mkdir()
    PRICES_PATH, REJECTED_PATH = DATA_DEST / "prices.parquet", DATA_DEST / "rejected.parquet"
    with zipfile.ZipFile(archive_path) as archive:
        PRICES_PATH.write_bytes(archive.read(price_member))
        REJECTED_PATH.write_bytes(archive.read(reject_member))
else:
    raise RuntimeError("Có nhiều snapshot: chọn PRICES_OVERRIDE và REJECTED_OVERRIDE.")
from src.forecast.kaggle_dlinear import audit_training_data

prices, rejected, prepared = audit_training_data(PRICES_PATH, REJECTED_PATH)
print("Model assets:", sorted(prepared), "clean rows:", len(prices), "rejected:", len(rejected))
for asset, data in prepared.items():
    print(asset, data.audit)

## 4. Tests source tổng hợp và CLI trước khi train
Không cần snapshot riêng cho unit tests. Full repository CI sử dụng requirements-dev.txt; đây là tập tests DLinear và Feature Builder cần cho training.

In [ ]:
subprocess.run([sys.executable, "-m", "src.forecast.train_dlinear", "--help"], check=True)
tests = ["tests/test_dlinear.py", "tests/test_kaggle_package.py"]
if (REPO_ROOT / "tests/test_features_build.py").is_file():
    tests.append("tests/test_features_build.py")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        *tests,
        "--basetemp",
        str(WORKING_ROOT / "cpaws-unit-test-temp"),
    ],
    check=True,
)

## 5. Cấu hình thực nghiệm
Các defaults không phải cấu hình tối ưu. Điền commit nguồn thật khi train chính thức. None cho phép thử nghiệm nhưng artifacts không đủ điều kiện tích hợp chính thức. Không early stopping trên 2019.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
SEED = 7
DEVICE = "cpu"  # "auto" hoặc "cuda" khi bật GPU
OPTIMIZER = "adam"
THREADS = 1
SOURCE_COMMIT = None
SOURCE_COMMIT = (SOURCE_COMMIT.strip() or None) if SOURCE_COMMIT is not None else None
MODEL_VERSION = "dlinear-" + uuid.uuid4().hex[:12]
MODELS_DIR = WORKING_ROOT / "models"
ASSETS_TO_TRAIN = ("gold", "silver", "brent", "wti")

## 6. Training bốn model độc lập bằng CLI repository
Console in số valid/excluded và loss từng epoch. Không phải validation 2019 hoặc accuracy.

In [ ]:
for asset in ASSETS_TO_TRAIN:
    command = [
        sys.executable,
        "-m",
        "src.forecast.train_dlinear",
        "--asset",
        asset,
        "--prices",
        str(PRICES_PATH),
        "--rejected",
        str(REJECTED_PATH),
        "--output-dir",
        str(MODELS_DIR),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--learning-rate",
        str(LEARNING_RATE),
        "--seed",
        str(SEED),
        "--device",
        DEVICE,
        "--optimizer",
        OPTIMIZER,
        "--threads",
        str(THREADS),
        "--model-version",
        MODEL_VERSION,
    ]
    if SOURCE_COMMIT:
        command += ["--source-commit", SOURCE_COMMIT]
    subprocess.run(command, check=True)

## 7. Xác minh 16 artifacts và NumPy/PyTorch parity; ZIP kết quả hiện tại

In [ ]:
import json

import torch

from src.forecast.dlinear_numpy import DLinearNumpy
from src.forecast.kaggle_dlinear import verify_trained_artifact

torch.set_num_threads(THREADS)
outputs = []
for asset in ASSETS_TO_TRAIN:
    directory = MODELS_DIR / asset / MODEL_VERSION
    for filename in ("weights.npz", "config.json", "scaler.json", "manifest.json"):
        path = directory / filename
        assert path.is_file(), path
        outputs.append(path)
    manifest = json.loads((directory / "manifest.json").read_text())
    assert manifest["asset"] == asset and manifest["max_training_target_date"] <= "2018-12-31"
    assert manifest["commit"] == SOURCE_COMMIT
    assert manifest["integration_eligible"] == bool(SOURCE_COMMIT)
    parity = verify_trained_artifact(directory, prepared[asset])
    assert DLinearNumpy.load(directory).predict(prepared[asset].X[:1])[0].shape == (7, 7)
    print(asset, "parity:", parity, "eligible:", manifest["integration_eligible"])
assert len(outputs) == 16
OUTPUT_ZIP = WORKING_ROOT / "trained-dlinear-models.zip"
if OUTPUT_ZIP.exists():
    raise FileExistsError("Lưu/đổi tên ZIP kết quả cũ trước khi đóng gói lại.")
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.relative_to(WORKING_ROOT).as_posix())
print("Download:", OUTPUT_ZIP)
try:
    from IPython.display import FileLink, display

    display(FileLink(str(OUTPUT_ZIP)))
except ImportError:
    pass